# Microbatch Benchmark Analysis

This notebook analyses profiler output from the microbatch benchmark across two configurations:

- **Baseline** – vanilla Storm topology (no SGX / Teaclave overhead)
- **Confidential (SGX)** – topology running inside Intel SGX enclaves via Teaclave

## Sections
- **Configuration** – paths, filters, colours
- **Load Run Catalog** – discover runs from both directories, parse `params.txt`, read profiler CSVs
- **Run Status Overview** – completion / failure counts per batch size
- **A – Throughput Analysis** – mean ± std throughput (GB/s) per batch size, broken out by parallelism and worker count
- **B – Baseline vs Confidential Comparison** – side-by-side and overhead ratio per (parallelism, n_workers) config (skipped if no baseline data)
- **C – Scaling Analysis** – throughput and duration vs batch size; parallelism / worker-count scaling curves
- **D – Run Catalog Table** – full listing of discovered runs

## Directory structure expected
```
data/runs/<label>/<timestamp>/n=<N>/<global-run-id>/<single-run-id>/
    params.txt
    profiler/<worker-IP>/microbatch-*-run<ID>.csv
```
An older single-level layout is also supported:
```
data/runs/<label>/<timestamp>/n=<N>/<global-run-id>/
    params.txt
    profiler/<worker-IP>/microbatch-*-run<ID>.csv   # may contain multiple batch rows
```

In [ ]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## Configuration

In [ ]:
# ---- Data directories (relative to this notebook in benchmark/) ----
CONF_RUNS_DIR = Path("../examples/microbatch-benchmark-confidential/data/runs")
BASE_RUNS_DIR  = Path("../examples/microbatch-benchmark-baseline/data/runs")

# ---- Filters ----
FILTER_LABELS: list[str]     = ["microbatch-paper"]
# Pin to specific sweep timestamps (empty = include all)
FILTER_TIMESTAMPS: list[str] = []

# ---- Output ----
SAVE_PLOTS  = False
OUTPUT_DIR  = Path("plots/microbatch")
PLOT_FORMAT = "pdf"

# ---- Colours ----
COLOR_BASELINE     = "#4472C4"   # blue
COLOR_CONFIDENTIAL = "#ED7D31"   # orange

if SAVE_PLOTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load Run Catalog

Discover all runs from both benchmark directories, parse `params.txt`, read profiler CSVs,
and assemble a unified catalog with one row per completed batch measurement.

In [ ]:
# ---- params.txt parsing ----
_BOOL_FIELDS  = {"completed", "oom", "fatal"}
_INT_FIELDS   = {"sub_run_id", "run_id", "parallelism", "runs_per_size",
                  "batches_reported", "expected_batches", "producers"}
_FLOAT_FIELDS = {"batch_size_gb", "completion_timeout_ms"}


def _parse_params(path: Path) -> dict[str, Any] | None:
    if not path.is_file():
        return None
    result: dict[str, Any] = {}
    for line in path.read_text().strip().splitlines():
        line = line.strip()
        if not line or "=" not in line:
            continue
        k, _, v = line.partition("=")
        k, v = k.strip(), v.strip()
        if k in _BOOL_FIELDS:
            result[k] = v.lower() == "true"
        elif k in _INT_FIELDS:
            try:
                result[k] = int(v)
            except ValueError:
                result[k] = v
        elif k in _FLOAT_FIELDS:
            try:
                result[k] = float(v)
            except ValueError:
                result[k] = v
        else:
            result[k] = v
    return result or None


def _load_profiler_csv(profiler_dir: Path) -> pd.DataFrame | None:
    """Return the first non-empty microbatch CSV found under profiler_dir."""
    for f in sorted(profiler_dir.rglob("*.csv")):
        try:
            df = pd.read_csv(f)
            if not df.empty:
                return df
        except Exception:
            pass
    return None


def _n_workers(n_dir: Path) -> int:
    m = re.match(r"n=(\d+)$", n_dir.name)
    return int(m.group(1)) if m else 0


def discover_microbatch_runs(
    base_dir: Path,
    topology_type: str,
    labels: list[str] | None = None,
) -> pd.DataFrame:
    """Discover microbatch benchmark runs under base_dir.

    Handles two layouts:
    - **New**: <global-run-id>/<single-run-id>/params.txt + profiler/<IP>/<csv>
    - **Old**: <global-run-id>/params.txt + profiler/<IP>/<csv>  (multi-batch CSV)
    """
    if not base_dir.is_dir():
        print(f"  INFO: Directory not found, skipping: {base_dir}")
        return pd.DataFrame()

    records: list[dict[str, Any]] = []

    for label_dir in sorted(base_dir.iterdir()):
        if not label_dir.is_dir():
            continue
        if labels and label_dir.name not in labels:
            continue

        for ts_dir in sorted(label_dir.iterdir()):
            if not ts_dir.is_dir():
                continue

            for n_dir in sorted(ts_dir.iterdir()):
                if not n_dir.is_dir() or not n_dir.name.startswith("n="):
                    continue
                n_workers = _n_workers(n_dir)

                for global_run_dir in sorted(n_dir.iterdir()):
                    if not global_run_dir.is_dir():
                        continue

                    # Detect layout: new layout has sub-directories each with params.txt
                    sub_run_dirs = sorted(
                        d for d in global_run_dir.iterdir()
                        if d.is_dir() and (d / "params.txt").is_file()
                    )

                    if sub_run_dirs:
                        # New layout: one sub-directory per (batch_size, repetition)
                        for sub_dir in sub_run_dirs:
                            params   = _parse_params(sub_dir / "params.txt") or {}
                            csv_df   = _load_profiler_csv(sub_dir / "profiler")
                            has_data = csv_df is not None and not csv_df.empty
                            csv_row  = csv_df.iloc[0].to_dict() if has_data else {}

                            rep_m = re.search(r"rep(\d+)", sub_dir.name)
                            records.append({
                                "topology_type":  topology_type,
                                "label":          label_dir.name,
                                "grid_timestamp": ts_dir.name,
                                "n_workers":      n_workers,
                                "global_run_id":  global_run_dir.name,
                                "run_dir":        sub_dir.name,
                                "repetition":     int(rep_m.group(1)) if rep_m else None,
                                "has_data":       has_data,
                                **params,
                                **csv_row,
                            })
                    else:
                        # Old layout: params.txt at global-run-id level; CSV may have multiple batch rows
                        params = _parse_params(global_run_dir / "params.txt") or {}
                        if not params:
                            continue
                        csv_df   = _load_profiler_csv(global_run_dir / "profiler")
                        has_data = csv_df is not None and not csv_df.empty

                        if has_data:
                            for rep_idx, (_, csv_row) in enumerate(csv_df.iterrows()):
                                records.append({
                                    "topology_type":  topology_type,
                                    "label":          label_dir.name,
                                    "grid_timestamp": ts_dir.name,
                                    "n_workers":      n_workers,
                                    "global_run_id":  global_run_dir.name,
                                    "run_dir":        global_run_dir.name,
                                    "repetition":     rep_idx + 1,
                                    "has_data":       True,
                                    **params,
                                    **csv_row.to_dict(),
                                })
                        else:
                            records.append({
                                "topology_type":  topology_type,
                                "label":          label_dir.name,
                                "grid_timestamp": ts_dir.name,
                                "n_workers":      n_workers,
                                "global_run_id":  global_run_dir.name,
                                "run_dir":        global_run_dir.name,
                                "repetition":     None,
                                "has_data":       False,
                                **params,
                            })

    return pd.DataFrame(records)

In [ ]:
conf_df = discover_microbatch_runs(CONF_RUNS_DIR, "confidential", FILTER_LABELS)
base_df = discover_microbatch_runs(BASE_RUNS_DIR, "baseline",     FILTER_LABELS)
catalog = pd.concat([conf_df, base_df], ignore_index=True)

if FILTER_TIMESTAMPS:
    before  = len(catalog)
    catalog = catalog[catalog["grid_timestamp"].isin(FILTER_TIMESTAMPS)].copy()
    print(f"Timestamp filter applied: {before} → {len(catalog)} rows")

# ---- Normalise batch-size column ----
# New-format params.txt has 'batch_size_gb' (float); old-format CSV has 'size_gb' (float).
if "batch_size_gb" not in catalog.columns:
    catalog["batch_size_gb"] = np.nan
if "size_gb" in catalog.columns:
    catalog["batch_size_gb"] = catalog["batch_size_gb"].fillna(catalog["size_gb"])

# ---- Run status ----
catalog["status"] = "Completed"
if "completed" in catalog.columns:
    catalog.loc[catalog["completed"].fillna(True) == False, "status"] = "Timed out"
if "oom" in catalog.columns:
    catalog.loc[catalog["oom"].fillna(False) == True, "status"] = "OOM"
catalog.loc[~catalog["has_data"], "status"] = "No data"

# ---- Derived throughput metrics (only for rows with actual profiler data) ----
_has = catalog["has_data"]
if "duration_ms" in catalog.columns and "batch_size_gb" in catalog.columns:
    catalog["duration_s"] = catalog["duration_ms"] / 1000.0
    catalog.loc[_has, "throughput_gb_s"] = (
        catalog.loc[_has, "batch_size_gb"] / catalog.loc[_has, "duration_s"]
    )
if "n_records" in catalog.columns and "duration_s" in catalog.columns:
    catalog.loc[_has, "throughput_mrec_s"] = (
        catalog.loc[_has, "n_records"] / catalog.loc[_has, "duration_s"] / 1e6
    )

print(f"Loaded {len(catalog)} run entries")
print(f"  Topology types  : {dict(catalog['topology_type'].value_counts())}")
print(f"  Timestamps      : {sorted(catalog['grid_timestamp'].unique())}")
if "batch_size_gb" in catalog.columns:
    print(f"  Batch sizes (GB): {sorted(catalog['batch_size_gb'].dropna().unique())}")
if "parallelism" in catalog.columns:
    print(f"  Parallelism     : {sorted(catalog['parallelism'].dropna().unique())}")
if "n_workers" in catalog.columns:
    print(f"  Workers (n=)    : {sorted(catalog['n_workers'].dropna().unique())}")
print(f"  Status          : {dict(catalog['status'].value_counts())}")

## Run Status Overview

In [ ]:
if catalog.empty:
    print("No runs discovered.")
else:
    _grp_cols = [c for c in ["topology_type", "parallelism", "n_workers", "batch_size_gb", "grid_timestamp", "status"]
                 if c in catalog.columns]
    status_tbl = (
        catalog.groupby(_grp_cols)
        .size()
        .reset_index(name="count")
        .pivot_table(
            index=[c for c in _grp_cols if c != "status"],
            columns="status",
            values="count",
            fill_value=0,
        )
        .reset_index()
    )
    display(status_tbl)

---
# Section A: Throughput Analysis

Throughput is computed as `batch_size_gb / (duration_ms / 1000)` (GB/s) for every completed run.
Runs are grouped by `(topology_type, parallelism, n_workers)` so that different hardware
configurations appear as separate series, making it easy to see how parallelism and the number
of Storm worker nodes affect throughput.

In [ ]:
# Subset: completed runs with throughput data
_perf = catalog[catalog["has_data"] & catalog["throughput_gb_s"].notna()].copy()

if _perf.empty:
    print("No completed runs with profiler data found.")
else:
    sizes_gb = sorted(_perf["batch_size_gb"].dropna().unique())

    # ---- Build sorted list of unique (topology_type, parallelism, n_workers) groups ----
    # These become the legend series in all plots.
    _GRP_COLS = [c for c in ["topology_type", "parallelism", "n_workers"] if c in _perf.columns]
    _groups: list[tuple] = [
        tuple(row)
        for _, row in _perf[_GRP_COLS].drop_duplicates().sort_values(_GRP_COLS).iterrows()
    ]

    # ---- Per-group visual attributes ----
    _HATCHES = ["", "///", "\\\\\\", "|||", "---", "xxx"]
    _LSTYLES: list[Any] = ["-", "--", "-.", ":", (0, (3, 1, 1, 1)), (0, (5, 5))]
    _MARKERS = ["o", "s", "^", "D", "v", "P"]
    _TTYPE_COLORS = {"confidential": COLOR_CONFIDENTIAL, "baseline": COLOR_BASELINE}

    _ttype_idx: dict[str, int] = {}
    _group_color: dict[tuple, str]  = {}
    _group_hatch: dict[tuple, str]  = {}
    _group_ls:    dict[tuple, Any]  = {}
    _group_mk:    dict[tuple, str]  = {}
    _group_label: dict[tuple, str]  = {}

    for grp in _groups:
        ttype = grp[0]
        i = _ttype_idx.get(ttype, 0)
        _ttype_idx[ttype] = i + 1

        _group_color[grp] = _TTYPE_COLORS.get(ttype, "#888888")
        _group_hatch[grp] = _HATCHES[i % len(_HATCHES)]
        _group_ls[grp]    = _LSTYLES[i % len(_LSTYLES)]
        _group_mk[grp]    = _MARKERS[i % len(_MARKERS)]

        base_name = "Baseline" if "base" in ttype.lower() else "Confidential (SGX)"
        extras = []
        if "parallelism" in _GRP_COLS:
            p = grp[_GRP_COLS.index("parallelism")]
            extras.append(f"p={int(p)}" if pd.notna(p) else "p=?")
        if "n_workers" in _GRP_COLS:
            n = grp[_GRP_COLS.index("n_workers")]
            extras.append(f"n={int(n)}" if pd.notna(n) else "n=?")
        _group_label[grp] = f"{base_name} ({', '.join(extras)})" if extras else base_name

    # Legacy single-type dicts (still used in the baseline-comparison cells)
    ttypes = sorted(_perf["topology_type"].unique())
    _color = {t: _TTYPE_COLORS.get(t, "#888888") for t in ttypes}
    _label = {t: ("Baseline" if "base" in t.lower() else "Confidential (SGX)") for t in ttypes}

    print(f"Performance subset: {len(_perf)} rows  |  {len(sizes_gb)} batch size(s)  |  {len(_groups)} group(s)")
    for g in _groups:
        mask = pd.Series(True, index=_perf.index)
        for ci, col in enumerate(_GRP_COLS):
            mask &= (_perf[col] == g[ci])
        print(f"  {_group_label[g]}: {mask.sum()} runs")

### A1: Mean Throughput by Batch Size

Grouped bar chart with one series per `(topology_type, parallelism, n_workers)` configuration.
Error bars show ±1 standard deviation across repetitions.

In [ ]:
if _perf.empty:
    print("Skipped: no data.")
else:
    fig, ax = plt.subplots(figsize=(max(6, len(sizes_gb) * 1.8 * len(_groups)), 5))
    x = np.arange(len(sizes_gb))
    width = 0.7 / max(len(_groups), 1)

    for i, grp in enumerate(_groups):
        mask_grp = pd.Series(True, index=_perf.index)
        for ci, col in enumerate(_GRP_COLS):
            mask_grp &= (_perf[col] == grp[ci])

        means, stds, counts = [], [], []
        for s in sizes_gb:
            vals = _perf.loc[mask_grp & (_perf["batch_size_gb"] == s), "throughput_gb_s"].dropna().values
            means.append(np.mean(vals)  if len(vals) else np.nan)
            stds.append(np.std(vals, ddof=1) if len(vals) > 1 else 0.0)
            counts.append(len(vals))

        offset = (i - (len(_groups) - 1) / 2) * width
        bars = ax.bar(
            x + offset, means, width, yerr=stds,
            label=_group_label[grp],
            color=_group_color[grp],
            hatch=_group_hatch[grp],
            alpha=0.85, capsize=5,
            error_kw={"elinewidth": 1.5},
        )
        for bar, m, s, n in zip(bars, means, stds, counts):
            if np.isfinite(m):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    m + s + ax.get_ylim()[1] * 0.01,
                    f"{m:.4f}\n(n={n})",
                    ha="center", va="bottom", fontsize=8,
                )

    ax.set_xticks(x)
    ax.set_xticklabels([f"{s:g} GB" for s in sizes_gb])
    ax.set_xlabel("Batch size")
    ax.set_ylabel("Throughput (GB/s)")
    ax.set_title("A1 – Mean Throughput by Batch Size  (error bars = ±1 std)")
    ax.set_ylim(bottom=0)
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(fontsize=9)
    plt.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(OUTPUT_DIR / f"a1-throughput-bar.{PLOT_FORMAT}", dpi=150, bbox_inches="tight")
    plt.show()

### A2: Throughput Distribution (Box Plots)

Per-batch-size boxplot showing the spread of throughput across repetitions, broken out by
`(topology_type, parallelism, n_workers)`.  Individual run points are overlaid.

In [ ]:
if _perf.empty:
    print("Skipped: no data.")
else:
    group_width    = len(_groups) + 1
    positions_data: list[np.ndarray] = []
    box_positions:  list[float]      = []
    box_colors:     list[str]        = []
    box_hatches:    list[str]        = []
    tick_pos:       list[float]      = []
    tick_lbl:       list[str]        = []

    for gi, s in enumerate(sizes_gb):
        for ti, grp in enumerate(_groups):
            mask = _perf["batch_size_gb"] == s
            for ci, col in enumerate(_GRP_COLS):
                mask &= (_perf[col] == grp[ci])
            vals = _perf.loc[mask, "throughput_gb_s"].dropna().values
            pos  = gi * group_width + ti
            positions_data.append(vals if len(vals) > 0 else np.array([np.nan]))
            box_positions.append(pos)
            box_colors.append(_group_color[grp])
            box_hatches.append(_group_hatch[grp])
        tick_pos.append(gi * group_width + (len(_groups) - 1) / 2.0)
        tick_lbl.append(f"{s:g} GB")

    fig, ax = plt.subplots(figsize=(max(6, len(sizes_gb) * 1.8 * len(_groups) + 2), 5))
    bp = ax.boxplot(
        positions_data, positions=box_positions, patch_artist=True, widths=0.6,
        medianprops={"color": "black", "linewidth": 1.5},
        flierprops={"marker": "o", "markersize": 5, "alpha": 0.6},
    )
    for patch, col, hatch in zip(bp["boxes"], box_colors, box_hatches):
        patch.set_facecolor(col)
        patch.set_hatch(hatch)
        patch.set_alpha(0.7)

    for vals, pos, col in zip(positions_data, box_positions, box_colors):
        if len(vals) > 0 and not np.all(np.isnan(vals)):
            ax.scatter(
                np.full(len(vals), pos) + np.random.normal(0, 0.05, len(vals)),
                vals, color=col, s=25, alpha=0.8, zorder=3, edgecolors="white", linewidths=0.5,
            )

    ax.set_xticks(tick_pos)
    ax.set_xticklabels(tick_lbl)
    ax.set_ylabel("Throughput (GB/s)")
    ax.set_title("A2 – Throughput Distribution by Batch Size")
    ax.set_ylim(bottom=0)
    ax.grid(True, axis="y", alpha=0.3)
    handles = [
        mpatches.Patch(facecolor=_group_color[g], hatch=_group_hatch[g], alpha=0.7, label=_group_label[g])
        for g in _groups
    ]
    ax.legend(handles=handles, fontsize=9)
    plt.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(OUTPUT_DIR / f"a2-throughput-boxplot.{PLOT_FORMAT}", dpi=150, bbox_inches="tight")
    plt.show()

### A3: Batch Completion Time by Batch Size

Line plot of mean completion time (seconds) vs batch size, one line per configuration.
Linear scaling would appear as a straight line through the origin.

In [ ]:
if _perf.empty or "duration_s" not in _perf.columns:
    print("Skipped: no duration data.")
else:
    fig, ax = plt.subplots(figsize=(8, 5))
    for grp in _groups:
        mask = pd.Series(True, index=_perf.index)
        for ci, col in enumerate(_GRP_COLS):
            mask &= (_perf[col] == grp[ci])
        sub   = _perf[mask]
        means = sub.groupby("batch_size_gb")["duration_s"].mean()
        stds  = sub.groupby("batch_size_gb")["duration_s"].std(ddof=1).fillna(0)
        ax.errorbar(
            means.index, means.values, yerr=stds.values,
            marker=_group_mk[grp], linestyle=_group_ls[grp],
            color=_group_color[grp], label=_group_label[grp],
            capsize=5, linewidth=2, markersize=7,
        )
        for x_val, y_val in zip(means.index, means.values):
            ax.text(x_val, y_val * 1.04, f"{y_val:.0f}s", ha="center", va="bottom", fontsize=9)

    ax.set_xlabel("Batch size (GB)")
    ax.set_ylabel("Completion time (s)")
    ax.set_title("A3 – Batch Completion Time vs Batch Size")
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
    plt.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(OUTPUT_DIR / f"a3-duration-vs-size.{PLOT_FORMAT}", dpi=150, bbox_inches="tight")
    plt.show()

### A4: Throughput in Records/s

Same bar chart expressed in millions of records processed per second.

In [ ]:
if _perf.empty or "throughput_mrec_s" not in _perf.columns:
    print("Skipped: n_records not available.")
else:
    fig, ax = plt.subplots(figsize=(max(6, len(sizes_gb) * 1.8 * len(_groups)), 5))
    x = np.arange(len(sizes_gb))
    width = 0.7 / max(len(_groups), 1)

    for i, grp in enumerate(_groups):
        mask_grp = pd.Series(True, index=_perf.index)
        for ci, col in enumerate(_GRP_COLS):
            mask_grp &= (_perf[col] == grp[ci])

        means, stds = [], []
        for s in sizes_gb:
            vals = _perf.loc[mask_grp & (_perf["batch_size_gb"] == s), "throughput_mrec_s"].dropna().values
            means.append(np.mean(vals)  if len(vals) else np.nan)
            stds.append(np.std(vals, ddof=1) if len(vals) > 1 else 0.0)

        offset = (i - (len(_groups) - 1) / 2) * width
        bars = ax.bar(
            x + offset, means, width, yerr=stds,
            label=_group_label[grp],
            color=_group_color[grp],
            hatch=_group_hatch[grp],
            alpha=0.85, capsize=5,
            error_kw={"elinewidth": 1.5},
        )
        for bar, m, s in zip(bars, means, stds):
            if np.isfinite(m):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    m + s + ax.get_ylim()[1] * 0.01,
                    f"{m:.2f}",
                    ha="center", va="bottom", fontsize=9,
                )

    ax.set_xticks(x)
    ax.set_xticklabels([f"{s:g} GB" for s in sizes_gb])
    ax.set_xlabel("Batch size")
    ax.set_ylabel("Throughput (Mrec/s)")
    ax.set_title("A4 – Mean Throughput in Million Records/s (error bars = ±1 std)")
    ax.set_ylim(bottom=0)
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(fontsize=9)
    plt.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(OUTPUT_DIR / f"a4-throughput-records.{PLOT_FORMAT}", dpi=150, bbox_inches="tight")
    plt.show()

---
# Section B: Baseline vs Confidential Comparison

Direct comparison of throughput between topology types for matching `(batch_size_gb, parallelism, n_workers)`
configurations.  This section is skipped gracefully when no baseline data is available.

In [ ]:
_conf_perf = _perf[_perf["topology_type"] == "confidential"]
_base_perf = _perf[_perf["topology_type"] == "baseline"]

if _base_perf.empty:
    print("Section B skipped: no baseline data available yet.")
    print(f"  Add baseline runs to: {BASE_RUNS_DIR}")
else:
    print(f"Confidential runs: {len(_conf_perf)}")
    print(f"Baseline runs    : {len(_base_perf)}")

    # Find (parallelism, n_workers) configurations present in BOTH topology types
    _cfg_cols = [c for c in ["parallelism", "n_workers"] if c in _perf.columns]
    if _cfg_cols:
        _conf_cfgs = set(tuple(r) for _, r in _conf_perf[_cfg_cols].drop_duplicates().iterrows())
        _base_cfgs = set(tuple(r) for _, r in _base_perf[_cfg_cols].drop_duplicates().iterrows())
        _shared_cfgs = sorted(_conf_cfgs & _base_cfgs)
        print(f"  Shared configs  : {[dict(zip(_cfg_cols, c)) for c in _shared_cfgs]}")
    else:
        _shared_cfgs = [()]
        _cfg_cols    = []

### B1 + B2: Throughput Comparison and Overhead Ratio

One figure per shared `(parallelism, n_workers)` configuration:
- **Left**: side-by-side throughput bars for baseline vs confidential
- **Right**: overhead ratio `confidential / baseline` (green ≥ 0.9, amber ≥ 0.5, red < 0.5)

In [ ]:
if _base_perf.empty:
    print("Skipped: no baseline data.")
else:
    _shared_sizes = sorted(
        set(_conf_perf["batch_size_gb"].dropna().unique())
        & set(_base_perf["batch_size_gb"].dropna().unique())
    )
    if not _shared_sizes:
        print("No matching batch sizes between baseline and confidential.")
    else:
        for cfg in (_shared_cfgs if _shared_cfgs else [()]):
            if cfg:
                cfg_dict  = dict(zip(_cfg_cols, cfg))
                cfg_label = ", ".join(f"{k}={int(v)}" for k, v in cfg_dict.items())
                _c = _conf_perf.copy()
                _b = _base_perf.copy()
                for col, val in cfg_dict.items():
                    _c = _c[_c[col] == val]
                    _b = _b[_b[col] == val]
            else:
                cfg_label, _c, _b = "", _conf_perf, _base_perf

            title_sfx = f" ({cfg_label})" if cfg_label else ""
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            x = np.arange(len(_shared_sizes))
            width = 0.35

            # Left: absolute throughput
            ax = axes[0]
            for i, (sub, ttype) in enumerate([(_c, "confidential"), (_b, "baseline")]):
                means = [sub.loc[sub["batch_size_gb"] == s, "throughput_gb_s"].mean() for s in _shared_sizes]
                stds  = [v if np.isfinite(v) else 0.0
                         for v in [sub.loc[sub["batch_size_gb"] == s, "throughput_gb_s"].std(ddof=1)
                                   for s in _shared_sizes]]
                col   = COLOR_CONFIDENTIAL if ttype == "confidential" else COLOR_BASELINE
                lbl   = ("Confidential (SGX)" if ttype == "confidential" else "Baseline") + title_sfx
                offset = (i - 0.5) * width
                ax.bar(x + offset, means, width, yerr=stds,
                       label=lbl, color=col, alpha=0.85, capsize=5, error_kw={"elinewidth": 1.5})
            ax.set_xticks(x)
            ax.set_xticklabels([f"{s:g} GB" for s in _shared_sizes])
            ax.set_xlabel("Batch size")
            ax.set_ylabel("Throughput (GB/s)")
            ax.set_title(f"B1 – Throughput: Baseline vs Confidential{title_sfx}")
            ax.set_ylim(bottom=0)
            ax.grid(True, axis="y", alpha=0.3)
            ax.legend(fontsize=9)

            # Right: overhead ratio confidential / baseline
            ax = axes[1]
            ratios = []
            for s in _shared_sizes:
                c_m = _c.loc[_c["batch_size_gb"] == s, "throughput_gb_s"].mean()
                b_m = _b.loc[_b["batch_size_gb"] == s, "throughput_gb_s"].mean()
                ratios.append(c_m / b_m if (np.isfinite(b_m) and b_m > 0) else np.nan)

            bar_cols = ["#2ecc71" if r >= 0.9 else ("#f39c12" if r >= 0.5 else "#e74c3c")
                        for r in ratios]
            bars = ax.bar(x, ratios, 0.5, color=bar_cols, alpha=0.85, edgecolor="black", linewidth=0.5)
            ax.axhline(1.0, color="black", linestyle="--", linewidth=1.2, label="No overhead (ratio=1)")
            for bar, r in zip(bars, ratios):
                if np.isfinite(r):
                    ax.text(bar.get_x() + bar.get_width() / 2, r + 0.01,
                            f"{r:.2f}×", ha="center", va="bottom", fontsize=10, fontweight="bold")
            ax.set_xticks(x)
            ax.set_xticklabels([f"{s:g} GB" for s in _shared_sizes])
            ax.set_xlabel("Batch size")
            ax.set_ylabel("Throughput ratio  (confidential / baseline)")
            ax.set_title(f"B2 – Overhead Ratio{title_sfx}")
            ax.set_ylim(bottom=0)
            ax.grid(True, axis="y", alpha=0.3)
            ax.legend(fontsize=9)

            plt.tight_layout()
            if SAVE_PLOTS:
                safe = cfg_label.replace(", ", "_").replace("=", "") if cfg_label else "all"
                fig.savefig(OUTPUT_DIR / f"b-comparison-{safe}.{PLOT_FORMAT}", dpi=150, bbox_inches="tight")
            plt.show()

### B3: Overhead Summary Table

In [ ]:
if _base_perf.empty:
    print("Skipped: no baseline data.")
else:
    _shared_sizes = sorted(
        set(_conf_perf["batch_size_gb"].dropna().unique())
        & set(_base_perf["batch_size_gb"].dropna().unique())
    )
    rows = []
    for cfg in (_shared_cfgs if _shared_cfgs else [()]):
        cfg_dict = dict(zip(_cfg_cols, cfg)) if cfg else {}
        for s in _shared_sizes:
            _c = _conf_perf[_conf_perf["batch_size_gb"] == s]
            _b = _base_perf[_base_perf["batch_size_gb"] == s]
            for col, val in cfg_dict.items():
                _c = _c[_c[col] == val]
                _b = _b[_b[col] == val]
            c_vals = _c["throughput_gb_s"].dropna().values
            b_vals = _b["throughput_gb_s"].dropna().values
            c_dur  = _c["duration_s"].dropna().values
            b_dur  = _b["duration_s"].dropna().values
            ratio  = (np.mean(c_vals) / np.mean(b_vals)
                      if len(c_vals) and len(b_vals) and np.mean(b_vals) > 0 else np.nan)

            def _fmt(vals, ddof=1):
                if not len(vals): return "—"
                return f"{np.mean(vals):.4f} ± {np.std(vals, ddof=ddof):.4f}" if len(vals) > 1 else f"{vals[0]:.4f}"

            def _fmt_s(vals, ddof=1):
                if not len(vals): return "—"
                return f"{np.mean(vals):.1f} ± {np.std(vals, ddof=ddof):.1f}" if len(vals) > 1 else f"{vals[0]:.1f}"

            row: dict[str, Any] = {"Batch size (GB)": s}
            row.update(cfg_dict)
            row.update({
                "Conf. tput (GB/s)": _fmt(c_vals),
                "Base. tput (GB/s)": _fmt(b_vals),
                "Conf. time (s)":    _fmt_s(c_dur),
                "Base. time (s)":    _fmt_s(b_dur),
                "Ratio (conf/base)": f"{ratio:.3f}" if np.isfinite(ratio) else "—",
            })
            rows.append(row)
    display(pd.DataFrame(rows))

---
# Section C: Scaling Analysis

Two angles:
1. **Batch-size scaling** – does throughput stay flat as the batch grows? (C1, C2)
2. **Parallelism / worker-count scaling** – does throughput grow linearly with more executors or nodes? (C3, C4)

### C1 + C2: Throughput and Duration vs Batch Size

In [ ]:
if _perf.empty or len(sizes_gb) < 2:
    print("Skipped: need at least 2 batch sizes for scaling analysis.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: throughput (GB/s) vs batch size — flat = linear scaling
    ax = axes[0]
    for grp in _groups:
        mask = pd.Series(True, index=_perf.index)
        for ci, col in enumerate(_GRP_COLS):
            mask &= (_perf[col] == grp[ci])
        sub   = _perf[mask]
        means = sub.groupby("batch_size_gb")["throughput_gb_s"].mean()
        stds  = sub.groupby("batch_size_gb")["throughput_gb_s"].std(ddof=1).fillna(0)
        ax.errorbar(
            means.index, means.values, yerr=stds.values,
            marker=_group_mk[grp], linestyle=_group_ls[grp],
            color=_group_color[grp], label=_group_label[grp],
            capsize=5, linewidth=2, markersize=7,
        )
    ax.set_xlabel("Batch size (GB)")
    ax.set_ylabel("Throughput (GB/s)")
    ax.set_title("C1 – Throughput vs Batch Size\n(flat curve = linear scaling)")
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

    # Right: log-log duration vs batch size — slope 1 = linear scaling
    ax = axes[1]
    for grp in _groups:
        mask = pd.Series(True, index=_perf.index)
        for ci, col in enumerate(_GRP_COLS):
            mask &= (_perf[col] == grp[ci])
        sub   = _perf[mask]
        means = sub.groupby("batch_size_gb")["duration_s"].mean()
        ax.loglog(
            means.index, means.values,
            marker=_group_mk[grp], linestyle=_group_ls[grp],
            color=_group_color[grp], label=_group_label[grp],
            linewidth=2, markersize=7,
        )

    _xs = np.array(sizes_gb, dtype=float)
    if len(_xs) >= 2:
        _ref_y0 = _perf["duration_s"].dropna().min() * 0.5
        ax.loglog(_xs, _ref_y0 * (_xs / _xs[0]), "k--", linewidth=1, alpha=0.5, label="slope = 1 (linear)")

    ax.set_xlabel("Batch size (GB)")
    ax.set_ylabel("Completion time (s)")
    ax.set_title("C2 – Duration vs Batch Size (log-log)\n(slope 1 = linear scaling)")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=9)

    plt.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(OUTPUT_DIR / f"c12-batch-scaling.{PLOT_FORMAT}", dpi=150, bbox_inches="tight")
    plt.show()

### C3 + C4: Parallelism / Worker-count Scaling

How does throughput change when `parallelism` (Storm executor count) or `n_workers` (number of
Storm supervisor nodes) grows?  Ideal linear scaling would double throughput when doubling
either parameter.  The efficiency plot on the right normalises to the smallest configuration.

In [ ]:
_scale_candidates = [
    (col, "Parallelism (executors)" if col == "parallelism" else "Number of workers (n=)")
    for col in ["parallelism", "n_workers"]
    if col in _perf.columns and _perf[col].nunique() > 1
]

if not _scale_candidates:
    print("C3/C4 skipped: only one parallelism / n_workers value in the data.")
    print("  Re-run the benchmark with different --parallelism or worker counts to enable this plot.")
else:
    _STYLE_CYCLE = ["-o", "--s", "-^", "--D", "-v", "--P"]

    for scale_col, col_display in _scale_candidates:
        scale_vals = sorted(_perf[scale_col].dropna().unique())
        min_x      = scale_vals[0]

        # Series: one line per (topology_type, batch_size_gb)
        _combo_cols = [c for c in ["topology_type", "batch_size_gb"] if c in _perf.columns]
        _combos = [
            tuple(row)
            for _, row in _perf[_combo_cols].drop_duplicates().sort_values(_combo_cols).iterrows()
        ]

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Left: absolute throughput vs scale_col
        ax = axes[0]
        _baseline_vals: dict[tuple, float] = {}
        for si, combo in enumerate(_combos):
            mask = pd.Series(True, index=_perf.index)
            for ci, col in enumerate(_combo_cols):
                mask &= (_perf[col] == combo[ci])
            grp_data = _perf[mask].groupby(scale_col)["throughput_gb_s"]
            means = grp_data.mean()
            stds  = grp_data.std(ddof=1).fillna(0)
            ttype = combo[0]
            s_gb  = combo[1] if len(combo) > 1 else None
            col_  = COLOR_BASELINE if "base" in ttype.lower() else COLOR_CONFIDENTIAL
            lbl   = ("Baseline" if "base" in ttype.lower() else "Confidential (SGX)")
            if s_gb is not None:
                lbl += f" {s_gb:g} GB"
            fmt   = _STYLE_CYCLE[si % len(_STYLE_CYCLE)]
            ax.errorbar(means.index, means.values, yerr=stds.values,
                        fmt=fmt, color=col_, label=lbl, capsize=5, linewidth=2, markersize=7)
            if min_x in means.index:
                _baseline_vals[combo] = means[min_x]

        # Ideal linear reference anchored at the global mean at min_x
        if _baseline_vals:
            _anchor = np.nanmean(list(_baseline_vals.values()))
            _xs = np.array(scale_vals, dtype=float)
            ax.plot(_xs, _anchor * _xs / min_x, "k--", linewidth=1, alpha=0.5, label="Ideal linear")

        ax.set_xlabel(col_display)
        ax.set_ylabel("Throughput (GB/s)")
        ax.set_title(f"C3 – Throughput vs {col_display}\n(dashed = ideal linear scaling)")
        ax.set_ylim(bottom=0)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=9)

        # Right: parallel efficiency = actual / ideal (normalised to 1 at min_x)
        ax = axes[1]
        for si, combo in enumerate(_combos):
            mask = pd.Series(True, index=_perf.index)
            for ci, col in enumerate(_combo_cols):
                mask &= (_perf[col] == combo[ci])
            means = _perf[mask].groupby(scale_col)["throughput_gb_s"].mean()
            base_val = _baseline_vals.get(combo, np.nan)
            if not np.isfinite(base_val) or base_val == 0:
                continue
            efficiency = means / (base_val * means.index / min_x)
            ttype = combo[0]
            s_gb  = combo[1] if len(combo) > 1 else None
            col_  = COLOR_BASELINE if "base" in ttype.lower() else COLOR_CONFIDENTIAL
            lbl   = ("Baseline" if "base" in ttype.lower() else "Confidential (SGX)")
            if s_gb is not None:
                lbl += f" {s_gb:g} GB"
            fmt   = _STYLE_CYCLE[si % len(_STYLE_CYCLE)]
            ax.plot(efficiency.index, efficiency.values, fmt, color=col_, label=lbl, linewidth=2, markersize=7)

        ax.axhline(1.0, color="black", linestyle="--", linewidth=1, alpha=0.5, label="Perfect efficiency")
        ax.set_xlabel(col_display)
        ax.set_ylabel("Parallel efficiency  (actual / ideal)")
        ax.set_title(f"C4 – Parallel Efficiency vs {col_display}")
        ax.set_ylim(bottom=0)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=9)

        plt.tight_layout()
        if SAVE_PLOTS:
            fig.savefig(OUTPUT_DIR / f"c34-scaling-{scale_col}.{PLOT_FORMAT}", dpi=150, bbox_inches="tight")
        plt.show()

### C5: Per-run Timeline (Gantt)

Horizontal bars showing when each batch run started and ended relative to the earliest start
in the sweep.  Useful for spotting scheduling artefacts or overlap between runs.

In [ ]:
_timeline_cols = {"t_begin_epoch_ms", "t_end_epoch_ms", "batch_size_gb", "topology_type"}
if not _timeline_cols.issubset(set(_perf.columns)):
    print(f"Skipped: need columns {_timeline_cols - set(_perf.columns)}")
elif _perf.empty:
    print("Skipped: no data.")
else:
    _tl = _perf.dropna(subset=["t_begin_epoch_ms", "t_end_epoch_ms"]).copy()
    if _tl.empty:
        print("Skipped: no runs with timestamp data.")
    else:
        t0_ms = _tl["t_begin_epoch_ms"].min()
        _tl["t_start_s"] = (_tl["t_begin_epoch_ms"] - t0_ms) / 1000.0
        _tl["t_end_s"]   = (_tl["t_end_epoch_ms"]   - t0_ms) / 1000.0

        # Sort by group then start time
        sort_cols = [c for c in ["topology_type", "parallelism", "n_workers", "batch_size_gb", "t_start_s"]
                     if c in _tl.columns]
        _tl = _tl.sort_values(sort_cols).reset_index(drop=True)

        fig, ax = plt.subplots(figsize=(12, max(4, len(_tl) * 0.45)))
        ytick_pos, ytick_lbl = [], []

        for yi, (_, row) in enumerate(_tl.iterrows()):
            # Determine which group this row belongs to
            grp = tuple(row[c] for c in _GRP_COLS if c in _tl.columns)
            col = _group_color.get(grp, _color.get(row.get("topology_type", ""), "gray"))
            ax.barh(
                yi, row["t_end_s"] - row["t_start_s"], left=row["t_start_s"],
                height=0.6, color=col, alpha=0.8, edgecolor="white", linewidth=0.5,
            )
            size_str = f"{row['batch_size_gb']:g} GB" if pd.notna(row.get("batch_size_gb")) else "?"
            rep_str  = f"rep{int(row['repetition'])}" if pd.notna(row.get("repetition")) else ""
            grp_lbl  = _group_label.get(grp, row.get("topology_type", ""))
            ytick_pos.append(yi)
            ytick_lbl.append(f"{size_str}  {rep_str}  [{grp_lbl}]")

        ax.set_yticks(ytick_pos)
        ax.set_yticklabels(ytick_lbl, fontsize=8)
        ax.set_xlabel("Elapsed time since first run start (s)")
        ax.set_title("C5 – Batch Run Timeline (Gantt)")
        ax.grid(True, axis="x", alpha=0.3)
        handles = [
            mpatches.Patch(facecolor=_group_color[g], hatch=_group_hatch[g], alpha=0.8, label=_group_label[g])
            for g in _groups
        ]
        ax.legend(handles=handles, loc="upper right", fontsize=9)
        plt.tight_layout()
        if SAVE_PLOTS:
            fig.savefig(OUTPUT_DIR / f"c5-timeline.{PLOT_FORMAT}", dpi=150, bbox_inches="tight")
        plt.show()

---
# Section D: Run Catalog Table

Full listing of all discovered runs with their key metrics.

In [ ]:
_display_cols = [
    "topology_type", "grid_timestamp", "n_workers", "parallelism",
    "global_run_id", "batch_size_gb", "bytes_per_tuple",
    "repetition", "status",
    "duration_s", "throughput_gb_s", "throughput_mrec_s",
]
_display_cols = [c for c in _display_cols if c in catalog.columns]

display(
    catalog[_display_cols]
    .sort_values(["topology_type", "parallelism", "n_workers", "batch_size_gb", "repetition"],
                 key=lambda s: s.fillna(-1) if s.dtype.kind in "biufc" else s.fillna(""))
    .reset_index(drop=True)
    .round(4)
    .style
    .format(na_rep="—")
    .background_gradient(
        subset=[c for c in ["throughput_gb_s", "throughput_mrec_s"] if c in _display_cols],
        cmap="YlGn",
    )
)

### D2: Summary Statistics per (topology, parallelism, n_workers, batch_size)

In [ ]:
if _perf.empty:
    print("No completed runs.")
else:
    _grp_by = [c for c in ["topology_type", "parallelism", "n_workers", "batch_size_gb"] if c in _perf.columns]
    _metric_cols = [c for c in ["throughput_gb_s", "throughput_mrec_s", "duration_s"] if c in _perf.columns]
    summary = _perf.groupby(_grp_by)[_metric_cols].agg(["count", "mean", "std", "min", "max"]).round(4)
    summary.columns = ["_".join(c).strip() for c in summary.columns]
    display(summary.reset_index())